# Analysis: England k=30 NMF Topic Model

Exploratory analysis of the production England model. All analysis in one place for understanding the data before building the dashboard.

In [ ]:
import sys
sys.path.insert(0, "/workspaces/AM1_topic_modelling")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from model_pipeline.training.s05_nmf_training import get_top_words_per_topic

## 1. Load data

In [ ]:
df = pd.read_csv("/workspaces/AM1_topic_modelling/data/evaluation_outputs/topics_analysis_ready_eng.csv")
df["article_date"] = pd.to_datetime(df["article_date"], errors="coerce")
df["year"] = df["article_date"].dt.year
df["month"] = df["article_date"].dt.to_period("M").dt.to_timestamp()

print(f"Loaded: {df.shape}")
print(f"Date range: {df['article_date'].min()} to {df['article_date'].max()}")
print(f"Sources: {df['source'].nunique()} \u2014 {df['source'].value_counts().to_dict()}")
print(f"Topics: {df['topic_name'].nunique()}")

## 2. Topic distribution

In [ ]:
topic_counts = df["topic_name"].value_counts()
topic_pct = (topic_counts / len(df) * 100).round(1)

topic_df = pd.DataFrame({"count": topic_counts, "pct": topic_pct})
print(topic_df.to_string())
print(f"\nTotal articles: {len(df)}")
print(f"Range: {topic_pct.min()}% \u2013 {topic_pct.max()}%")

fig, ax = plt.subplots(figsize=(12, 8))
topic_counts.plot(kind="barh", ax=ax)
ax.set_xlabel("Number of articles")
ax.set_title("Topic distribution \u2014 England k=30")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("/workspaces/AM1_topic_modelling/data/evaluation_outputs/topic_distribution_eng_k30.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Source concentration per topic

In [ ]:
ct = pd.crosstab(df["source"], df["topic_name"], normalize="columns").round(2)
print("Source distribution per topic (column-normalised):")
print(ct.to_string())

# Summary: single-source topics
for topic in ct.columns:
    top_source = ct[topic].idxmax()
    top_pct = ct[topic].max()
    if top_pct > 0.90:
        print(f"  SINGLE-SOURCE: {topic} \u2014 {top_source} {top_pct:.0%}")

In [ ]:
fig, ax = plt.subplots(figsize=(20, 6))
sns.heatmap(ct, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax, linewidths=0.5)
ax.set_title("Source \u00d7 Topic Heatmap (normalised) \u2014 England k=30")
ax.set_ylabel("Source")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("/workspaces/AM1_topic_modelling/data/evaluation_outputs/source_topic_heatmap_eng_k30.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Trends over time \u2014 monthly topic attention

In [ ]:
monthly = df.groupby(["month", "topic_name"]).size().reset_index(name="count")

# Top 10 topics by total count
top10 = df["topic_name"].value_counts().head(10).index.tolist()

fig, ax = plt.subplots(figsize=(14, 7))
for topic in top10:
    topic_monthly = monthly[monthly["topic_name"] == topic]
    ax.plot(topic_monthly["month"], topic_monthly["count"], label=topic, marker=".")

# Election marker (July 2024)
ax.axvline(pd.Timestamp("2024-07-04"), color="red", linestyle="--", alpha=0.7, label="UK Election July 2024")

ax.set_xlabel("Month")
ax.set_ylabel("Articles per month")
ax.set_title("Monthly topic attention \u2014 Top 10 topics \u2014 England k=30")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig("/workspaces/AM1_topic_modelling/data/evaluation_outputs/trends_eng_k30.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
election_date = pd.Timestamp("2024-07-04")
df["period"] = df["article_date"].apply(lambda x: "post_election" if x >= election_date else "pre_election")

pre = df[df["period"] == "pre_election"]["topic_name"].value_counts()
post = df[df["period"] == "post_election"]["topic_name"].value_counts()

comparison = pd.DataFrame({
    "pre_count": pre,
    "post_count": post,
}).fillna(0).astype(int)

comparison["pre_rank"] = comparison["pre_count"].rank(ascending=False).astype(int)
comparison["post_rank"] = comparison["post_count"].rank(ascending=False).astype(int)
comparison["rank_change"] = comparison["pre_rank"] - comparison["post_rank"]

print("Pre/post election topic rank shifts:")
print(comparison.sort_values("rank_change", ascending=False).to_string())

print(f"\nBiggest risers (post-election):")
print(comparison.nlargest(5, "rank_change")[["pre_rank", "post_rank", "rank_change"]])

print(f"\nBiggest fallers (post-election):")
print(comparison.nsmallest(5, "rank_change")[["pre_rank", "post_rank", "rank_change"]])

## 5. Contestability scores \u2014 dominant weight distribution

In [ ]:
weights = df["dominant_topic_weight"]

print(f"Dominant weight stats:")
print(f"  Min:    {weights.min():.4f}")
print(f"  Mean:   {weights.mean():.4f}")
print(f"  Median: {weights.median():.4f}")
print(f"  Max:    {weights.max():.4f}")
print(f"  Std:    {weights.std():.4f}")

# How many articles have low confidence?
thresholds = [0.05, 0.10, 0.15, 0.20]
for t in thresholds:
    n = (weights < t).sum()
    print(f"  Articles with weight < {t}: {n} ({n/len(df)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(weights, bins=50, edgecolor="black", alpha=0.7)
axes[0].axvline(weights.mean(), color="red", linestyle="--", label=f"Mean: {weights.mean():.3f}")
axes[0].set_xlabel("Dominant topic weight")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of dominant topic weights")
axes[0].legend()

# Box plot by topic (top 10)
top10_names = df["topic_name"].value_counts().head(10).index
df_top10 = df[df["topic_name"].isin(top10_names)]
df_top10.boxplot(column="dominant_topic_weight", by="topic_name", ax=axes[1], rot=45)
axes[1].set_title("Dominant weight by topic (top 10)")
axes[1].set_xlabel("")
axes[1].set_ylabel("Dominant topic weight")
plt.suptitle("")

plt.tight_layout()
plt.savefig("/workspaces/AM1_topic_modelling/data/evaluation_outputs/contestability_eng_k30.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Top articles per topic

In [ ]:
def explore_topic(topic_name, n=5):
    """Show top N articles for a given topic, ranked by dominant weight."""
    topic_df = df[df["topic_name"] == topic_name].nlargest(n, "dominant_topic_weight")
    
    print(f"TOPIC: {topic_name} ({len(df[df['topic_name'] == topic_name])} articles)")
    print(f"{'='*120}\n")
    
    for rank, (_, row) in enumerate(topic_df.iterrows(), 1):
        title = row.get("title", "No title")
        source = row.get("source", "Unknown")
        date = str(row.get("article_date", ""))[:10]
        weight = row["dominant_topic_weight"]
        text = str(row.get("text_clean", ""))[:500]
        
        print(f"[{rank}] weight={weight:.4f} | {source} | {date}")
        print(f"    {title}")
        print(f"    {text}...\n")

# Example usage:
# explore_topic("child_and_family_welfare")
# explore_topic("ofsted_inspection_reform", n=10)
print("Available topics:")
for t in sorted(df["topic_name"].unique()):
    print(f"  {t}")

## 7. Summary statistics

In [ ]:
print("=" * 60)
print("ENGLAND k=30 MODEL SUMMARY")
print("=" * 60)
print(f"Articles:          {len(df)}")
print(f"Topics:            {df['topic_name'].nunique()}")
print(f"Sources:           {df['source'].nunique()}")
print(f"Date range:        {df['article_date'].min().date()} to {df['article_date'].max().date()}")
print(f"Largest topic:     {df['topic_name'].value_counts().index[0]} ({df['topic_name'].value_counts().iloc[0]} articles, {df['topic_name'].value_counts().iloc[0]/len(df)*100:.1f}%)")
print(f"Smallest topic:    {df['topic_name'].value_counts().index[-1]} ({df['topic_name'].value_counts().iloc[-1]} articles, {df['topic_name'].value_counts().iloc[-1]/len(df)*100:.1f}%)")
print(f"Mean dom. weight:  {df['dominant_topic_weight'].mean():.4f}")
print(f"Single-source (>90%): {sum(1 for t in ct.columns if ct[t].max() > 0.90)}/30")
print("=" * 60)